In [1]:
!pip install rdflib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 8.9 MB/s  0:00:00


In [11]:
import pandas as pd
import urllib.parse
from rdflib import Graph, URIRef, Literal, Namespace, RDF

# 1. Initialize Graph and Namespace
g = Graph()
IOT = Namespace("http://example.org/iot#")

# 2. Define Ontology Properties
connects_to = IOT.connectsTo
has_protocol = IOT.hasProtocol
has_port = IOT.hasPort
has_threat_label = IOT.hasThreatLabel

# 3. Load your 2000-row CSV
df = pd.read_csv('iot23_sample.csv', on_bad_lines='skip')

# 4. Iterate through rows to create RDF triples
for index, row in df.iterrows():
    try:
        # Get raw IP strings, convert to string, remove hidden spaces
        raw_src = str(row['id.orig_h']).strip()
        raw_dst = str(row['id.resp_h']).strip()

        # Skip empty rows or pandas 'nan' values
        if raw_src == 'nan' or raw_dst == 'nan' or not raw_src:
            continue

        # Replace dots with underscores and make 100% URL-safe
        src_ip = urllib.parse.quote(raw_src.replace('.', '_'))
        dst_ip = urllib.parse.quote(raw_dst.replace('.', '_'))

        # Create valid URIs
        src_node = URIRef(IOT + f"IP_{src_ip}")
        dst_node = URIRef(IOT + f"IP_{dst_ip}")

        # Add Type (Class)
        g.add((src_node, RDF.type, IOT.Device))
        g.add((dst_node, RDF.type, IOT.NetworkNode))

        # Add Connections (Object Properties)
        g.add((src_node, connects_to, dst_node))

        # Clean and Add Port and Protocol
        port_val = str(row['id.resp_p']).strip()
        proto_val = str(row['proto']).strip()
        g.add((dst_node, has_port, Literal(port_val)))
        g.add((dst_node, has_protocol, Literal(proto_val)))

        # Clean and Add Threat Label
        label = str(row['label']).strip()
        if label != '-' and label != 'Benign' and label != 'nan':
            g.add((src_node, has_threat_label, Literal(label)))

    except Exception as e:
        print(f"Skipped row {index} due to error: {e}")
        continue

# 5. Export to Turtle format
g.serialize(destination='iot_knowledge_graph.ttl', format='turtle')
print(f"Success! Generated {len(g)} RDF triples and saved to iot_knowledge_graph.ttl")

Success! Generated 8001 RDF triples and saved to iot_knowledge_graph.ttl


In [12]:
from google.colab import files
files.download('iot_knowledge_graph.ttl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>